# TargetGym quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YannBerthelot/TargetGym/blob/main/notebooks/quickstart.ipynb)

**TargetGym** is a set of JAX environments for *target MDPs*: tasks where you
reach a setpoint and then **hold it**, instead of reaching a goal once and
stopping. That is what most industrial control looks like.

This notebook runs on a free CPU runtime in about a minute. It covers:

1. running the tuned controller that ships with every environment
2. seeing what it scores, so you know what a learned policy has to beat
3. stepping 256 environments at once under `jit`, `vmap` and `scan`
4. using the Gymnasium interface with your existing tools

Docs: <https://github.com/YannBerthelot/TargetGym>

In [ ]:
!pip install -q target-gym

In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import target_gym
from target_gym.registry import REGISTRY, GROUPS
from target_gym.runners.runners import baseline_policy

print(f"target-gym {target_gym.__version__}")
print(f"{len(REGISTRY)} environments across {len(GROUPS)} families")
for key, label in GROUPS.items():
    names = [n for n, s in REGISTRY.items() if s.group == key]
    print(f"  {label:20s} {len(names):2d}  {', '.join(names[:4])}"
          f"{' ...' if len(names) > 4 else ''}")

## 1. Every environment ships a tuned controller

A benchmark whose only reference point is a random policy tells you an agent
learned *something*. One with a tuned PID tells you whether it learned
anything **useful**. Here is the 2D aircraft holding a commanded altitude.

In [ ]:
spec = REGISTRY["plane"]
env = spec.make_env()
params = spec.params_cls(max_steps_in_episode=600)

pid = spec.make_pid()
pid.reset()

obs, state = env.reset(jax.random.PRNGKey(0), params)

altitude, target = [], []
for t in range(int(params.max_steps_in_episode)):
    # Record *before* stepping: see the note below on auto-reset.
    altitude.append(float(state.z))
    target.append(float(state.target_altitude))

    action = np.atleast_1d(pid(np.asarray(obs)))
    obs, state, reward, terminated, truncated, info = env.step(
        jax.random.PRNGKey(t), state, action, params
    )
    if terminated or truncated:
        break

print(f"{len(altitude)} steps, final error {abs(altitude[-1] - target[-1]):.1f} m")

plt.figure(figsize=(9, 3.5))
plt.plot(altitude, label="altitude")
plt.plot(target, "--", label="commanded")
plt.xlabel("step")
plt.ylabel("altitude (m)")
plt.title("plane-v1, flown by its shipped PID")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

> **One gotcha worth knowing.** `env.step` follows the gymnax convention and
> **auto-resets** when an episode ends, so the state it hands back after a
> termination already belongs to the *next* episode. Record what you care
> about before you step, as above, or read `info['final_observation']`. Reading
> `state` after the loop gives you a fresh episode's altitude and a fresh
> target, which looks like a controller that suddenly stopped working.
>
> `terminated` means the plant reached a terminal state, such as a crash.
> `truncated` means the time limit ran out. Only `terminated` should zero your
> value bootstrap.

## 2. What does a policy have to beat?

Every environment is scored the same way: reward in `[0, 1]` per step, reaching
1 only while the target is held exactly. So returns are comparable to the
shipped baselines. Here is the CSTR, a small exothermic reactor, under three
controllers.

In [ ]:
spec = REGISTRY["cstr"]
env = spec.make_env()
params = spec.make_test_params()


def episode_return(policy, seed=0):
    obs, state = env.reset(jax.random.PRNGKey(seed), params)
    total = 0.0
    for t in range(int(params.max_steps_in_episode)):
        action = np.atleast_1d(policy(np.asarray(obs), state))
        obs, state, reward, terminated, truncated, _ = env.step(
            jax.random.PRNGKey(10_000 + t), state, action, params
        )
        total += float(reward)
        if terminated or truncated:
            break
    return total


# Both baselines are (obs, state) -> action, so one loop serves either and
# your own agent drops in beside them. The PID ignores `state` and the MPC
# ignores `obs`: it reads the true state, because it is a full-state ceiling
# rather than a peer to a policy that sees only what a plant instruments.
pid = baseline_policy(spec, "pid", params)
mpc = baseline_policy(spec, "mpc", params)

print(f"best constant action  {episode_return(lambda o, s: 0.0):7.1f}")
print(f"shipped PID           {episode_return(pid):7.1f}")
print(f"shipped MPC           {episode_return(mpc):7.1f}")

The MPC is there to act as an upper bound rather than a rival: it plans against
the model, so it shows roughly what is achievable. Where a shipped baseline is
*weak*, the docs say so and by how much, which matters more than the number.

## 3. It is JAX the whole way down

The environments are written so a rollout never leaves the accelerator. Here we
step 256 of them in parallel, 400 steps deep, inside a single compiled `scan`.

In [ ]:
spec = REGISTRY["cstr"]
env = spec.make_env()
params = spec.make_test_params()

N_ENVS, HORIZON = 256, 400
action_shape = env.action_space(params).shape or (1,)


def episode(key):
    keys = jax.random.split(key, HORIZON + 1)
    _, state = env.reset(keys[0], params)

    def step(state, k):
        _, state, reward, _, _, _ = env.step(
            k, state, jnp.zeros(action_shape), params
        )
        return state, reward

    _, rewards = jax.lax.scan(step, state, keys[1:])
    return rewards.sum()


batch = jax.jit(jax.vmap(episode))
keys = jax.random.split(jax.random.PRNGKey(0), N_ENVS)

batch(keys).block_until_ready()  # compile once, then measure
start = time.perf_counter()
returns = batch(keys).block_until_ready()
elapsed = time.perf_counter() - start

steps = N_ENVS * HORIZON
print(f"{steps:,} steps in {elapsed:.3f} s = {steps / elapsed / 1e6:.1f} M steps/s")
print(f"mean return {float(returns.mean()):.1f} over {N_ENVS} environments")

Throughput varies a lot across the suite, from a first-order lag to a kiln with
a half-hour transport delay, which is deliberate: you can iterate quickly on the
cheap ones and spend the sample budget where the dynamics are actually hard.

## 4. Or use it through Gymnasium

If your tooling expects Gymnasium, there is a wrapper, so stable-baselines3 and
friends work unchanged. You lose end-to-end GPU execution, since the agent then
lives outside JAX, but nothing else changes.

In [ ]:
from target_gym import GymnasiumPlane

gym_env = GymnasiumPlane()
obs, info = gym_env.reset(seed=0)

for _ in range(5):
    obs, reward, terminated, truncated, info = gym_env.step(
        gym_env.action_space.sample()
    )

print(f"observation {obs.shape}, reward {float(reward):.3f}")
print(f"action space {gym_env.action_space}")

## Where to go next

- **[Browse the environments](https://github.com/YannBerthelot/TargetGym/blob/main/docs/environments.md)**
  for shapes, tracked variables, baselines and the physics contract behind each one.
- **[Baselines](https://github.com/YannBerthelot/TargetGym/blob/main/docs/baselines.md)**
  for how the PID and MPC controllers are built and tuned.
- **[Reward shaping](https://github.com/YannBerthelot/TargetGym/blob/main/docs/reward-shaping.md)**
  for why the tracking reward has the shape it does.

When you publish a number, cite the **versioned** environment name, `plane-v1`
rather than `plane`. The version changes when the dynamics, the reward, the
parameters or the observation layout change, so a result quoted against it keeps
meaning what it meant.